# Install Packages (You'll have to restart session after this is completed)

In [ ]:
#@title Use CASSA Packages
# 1. Install necessary system-level C libraries
# (These are required for the C-compiler to build 21cmFAST)
!sudo apt-get update
!sudo apt-get install -y libfftw3-dev libgsl-dev
!pip install 21cmFAST==3.3.1
!pip install matplotlib==3.7.3 scipy==1.13.1 tools21cm==2.1.3

In [ ]:
#@title Or Use Latest Packages
# 1. Install necessary system-level C libraries
# (These are required for the C-compiler to build 21cmFAST)
!sudo apt-get update
!sudo apt-get install -y libfftw3-dev libgsl-dev
!pip install tools21cm
!pip install 21cmFAST

In [ ]:
#@title Use Google Drive
cdir = "/content/drive/MyDrive/astro-colab"
%cd {cdir}
wdir = !pwd
assert cdir == wdir[0], "Please mount Google Drive using the Files icon in the left pane AND ensure the folder 'astro-colab' exists. Then try again."

# Imports and Utility Definitions

In [ ]:
#@title Import
import py21cmfast as p21c
import numpy as np
import tools21cm as t2c

import warnings
warnings.filterwarnings("ignore")
from pathlib import Path
import os
import sys

# matplotlib imports
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib import cm, colors
import ipywidgets as widgets
from ipywidgets import interact

# plotly imports and definitions
import plotly.graph_objects as go
from plotly.colors import sample_colorscale, get_colorscale, make_colorscale
from plotly.subplots import make_subplots
import plotly.express as px
def mincut_colorscale(scalename, min, max, cutoff, N=15):
  return make_colorscale(sample_colorscale(scalename, np.linspace(cutoff,max,N)/max, low=min, high=max))

def GET_Z_CONE_RANGE(z_min,z_max,dz=None,BOX_PARAM=50,BOX_LEN=50): # filenames will be dependent on BOX_PARAM AND BOX_LEN
  z_full_range = np.round(t2c.lightcone.redshifts_at_equal_comoving_distance(5,35,BOX_PARAM,BOX_LEN),4)
  z_range = np.array([z for z in z_full_range if z_min <= z <= z_max])
  if dz is None:
    return z_range
  # indices = range(0,len(z_range),max(1,(z_range[-1]-z_range[0])//step))
  z_linspace = np.linspace(z_min,z_max,int(max(1,(z_max-z_min)//dz)))
  indices = []
  for zx in z_linspace:
    for i, z in enumerate(z_range):
      if zx-z < 0:
        indices.append(i)
        break
  return z_range[indices]

def get_brightness_temp_box(box, box_len, redshift, hubble, matter, baryon, save=False):
    filename = f"dTb_BOX_PARAM_{box}_Z_PARAM_{redshift}.npy"

    if os.path.exists(filename):
      print(f"Loading {filename}.")
      return np.load(filename)

    initial_conditions = p21c.initial_conditions(
        # HII_DIM is the number of cells along the length of a box
        # box_len is the total length of a side of the box
        user_params={"HII_DIM": box, "BOX_LEN": box_len, "N_THREADS": 2},
        cosmo_params=p21c.CosmoParams(
            SIGMA_8=0.8,
            hlittle=hubble,
            OMm=matter,
            OMb=baryon
        ),
        random_seed=54321
    )

    perturbed_field = p21c.perturb_field(
        redshift=redshift,
        init_boxes=initial_conditions
    )

    ionized_field = p21c.ionize_box(
        perturbed_field=perturbed_field
    )

    brightness_temp = p21c.brightness_temperature(
        ionized_box=ionized_field,
        perturbed_field=perturbed_field
    )

    dTb_box = brightness_temp.brightness_temp
    if save: np.save(filename, dTb_box)
    # print(brightness_temp.astro_params)
    # print(brightness_temp.user_params)
    # Return the raw numpy array
    return dTb_box


In [ ]:
GET_Z_CONE_RANGE(7,12,0.2)

# Simulation

In [ ]:
#@title Box Simulation
BOX_PARAM = 50
BOX_LEN = BOX_PARAM
Z_STEP = 0.2
Z_PARAM_RANGE = GET_Z_CONE_RANGE(7,12+1,Z_STEP)
Z_validate = lambda z: z_val if (z_val:=np.ceil(z/Z_STEP)*Z_STEP) in Z_PARAM_RANGE else Z_PARAM_RANGE[-1]

# 1. Simulate and return a numpy array of a box of size NxNxN, N=BOX_PARAM
dTb_array = {
  Z_PARAM: get_brightness_temp_box(
    box=BOX_PARAM,
    box_len=BOX_LEN,
    redshift=Z_PARAM,
    hubble=0.69+(0 if print(f"Simulating z={Z_PARAM} ...") else 0),
    matter=0.31,
    baryon=0.02,
    save=True
) for Z_PARAM in Z_PARAM_RANGE}

dTb_max = np.nanmax([np.nanmax(dTb_array[z]) for z in Z_PARAM_RANGE])
dTb_min = np.nanmin([np.nanmin(dTb_array[z]) for z in Z_PARAM_RANGE])

# Plotly Version

## Brightness Temperature

In [ ]:
#@title Plot a specific slice of the box
def plot_tomographic_slice(box_data, box_len, redshift, slice_index, cmap="Turbo", slice_plane='z'):
    if not -box_data.shape[2] <= slice_index < box_data.shape[2]:
      raise ValueError(f"Slice index {slice_index} is out of bounds for axis of size {box_data.shape[2]}")

    # Select the specific slice
    if slice_plane.lower() == 'x':
      Tb_slice = box_data[slice_index, :, :]
    elif slice_plane.lower() == 'y':
      Tb_slice = box_data[:, slice_index, :]
    else:
      Tb_slice = box_data[:, :, slice_index]


    fig = go.Figure(data=go.Heatmap(
        z=Tb_slice,
        hoverinfo = "z",
        zmax = dTb_max,
        zmin = dTb_min,
        colorscale = cmap,
        ))

    fig.update_layout(
    yaxis_scaleanchor="x",
    width = 600, height=600,
    title=dict(
        text=f"Brightness Temperature Slice {slice_index} at Redshift z = {redshift}",
        y=0.91,
        x=0.5,
        xanchor="center",
        yanchor="middle"
    ),
    xaxis=dict(title=dict(text=r"x [Mpc]")),
    yaxis=dict(title=dict(text=r"y [Mpc]"))
    )
    fig.show()

    fig.update_layout(
    autosize=True,
    width=None,
    height=None
)
    fig.write_html(f"htmls/heatmap_slice_{slice_index}_z_{redshift}.html")

Z_PARAM = 10
Z_PARAM = Z_validate(Z_PARAM)
plot_tomographic_slice(
    box_data=dTb_array[Z_PARAM],
    box_len=BOX_PARAM,
    redshift=Z_PARAM,
    slice_index=0
)

In [ ]:
#@title Brightness tempertature, Slide through a specific slice of the box
def plot_tomographic_slice(box_data, box_len, redshift, cmap="Turbo", slice_plane='z'):
    # Select the specific slice
    def select_slice(data, index, plane):
      if plane.lower() == 'x':
        data_slice = data[index, :, :]
      elif plane.lower() == 'y':
        data_slice = data[:, index, :]
      else:
        data_slice = data[:, :, index]
      return data_slice

    maketitle = lambda index, z: f"Brightness Temperature Slice {index} – Redshift z = {z}"

    fig = go.Figure()

    for slice_index in range(BOX_PARAM):
      fig.add_trace(go.Heatmap(
        z=select_slice(box_data, slice_index, slice_plane),
        hoverinfo = "z",
        zmax = dTb_max,
        zmin = dTb_min,
        colorscale = cmap,
    ))

    fig.data[0].visible = True

    sliders = [dict(
    active= box_len-1,
    # currentvalue={"prefix": "Frequency: "},
    pad={"t": 50},
    steps=[dict(
        method="update",
        args=[
            {"visible": [k == i for k in range(len(fig.data))]},
            {"title": dict(
                            text=maketitle(i, redshift),
                            y=0.91,
                            x=0.5,
                            xanchor="center",
                            yanchor="middle"
                        )}
        ],
        label=f"{i}",
    ) for i in range(len(fig.data))]
    )]

    fig.update_layout(
    yaxis_scaleanchor="x",
    width = 550, height=600,
    title=dict(
        text=maketitle(slice_index, redshift),
        y=0.91,
        x=0.5,
        xanchor="center",
        yanchor="middle"
    ),
    xaxis=dict(title=dict(text=r"x [Mpc]")),
    yaxis=dict(title=dict(text=r"y [Mpc]")),
    sliders=sliders
    )

    fig.show()
    fig.update_layout(
    autosize=True,
    width=None,
    height=None
)
    fig.write_html(f"htmls/heatmap_slice_slider_z_{redshift}.html")

Z_PARAM = 20
Z_PARAM = Z_validate(Z_PARAM)
plot_tomographic_slice(
    box_data=dTb_array[Z_PARAM],
    box_len=BOX_PARAM,
    redshift=Z_PARAM
)

In [ ]:
#@title Mean Brightness Temperature of the box, Slide through Redshift
def plot_tomographic_slice(boxes, box_len, redshift_range, cmap="Turbo", slice_plane='z'):
    # Select the specific slice
    def select_slice(box, index, plane):
      if plane.lower() == 'x':
        box_slice = box[index, :, :]
      elif plane.lower() == 'y':
        box_slice = box[:, index, :]
      else:
        box_slice = box[:, :, index]
      return box_slice

    def box_mean(box):
        mean = np.array([[np.mean(box[i, j, :]) for j in range(len(box[i]))] for i in range(len(box))])
        return mean

    maketitle = lambda index, z: f"Mean Brightness Temperature at redshift z={z}"

    fig = go.Figure()

    slice_index = 0
    for redshift in redshift_range:
      fig.add_trace(go.Heatmap(
        # z=select_slice(boxes[redshift], slice_index, slice_plane),
        z=box_mean(boxes[redshift]),
        hoverinfo = "z", # this is z-axis, not redshift
        zmax = dTb_max,
        zmin = dTb_min,
        colorscale = cmap,
    ))

    # fig.data[0].visible = True

    sliders = [dict(
    active=len(redshift_range)-1,
    # currentvalue={"prefix": "Frequency: "},
    pad={"t": 50},
    steps=[dict(
        method="update",
        args=[
            {"visible": [_z == z for _z in redshift_range]},
            {"title": dict(
                            text=maketitle(slice_index, z),
                            y=0.91,
                            x=0.5,
                            xanchor="center",
                            yanchor="middle"
                        )}
        ],
        label=f"z={z}",
    ) for z in redshift_range]
    )]

    fig.update_layout(
    yaxis_scaleanchor="x",
    width = 550, height=600,
    title=dict(
        text=maketitle(slice_index, redshift),
        y=0.91,
        x=0.5,
        xanchor="center",
        yanchor="middle"
    ),
    xaxis=dict(title=dict(text=r"x [Mpc]")),
    yaxis=dict(title=dict(text=r"y [Mpc]")),
    sliders=sliders
    )

    fig.show()
    return fig

fig = plot_tomographic_slice(
    boxes=dTb_array,
    box_len=BOX_PARAM,
    redshift_range=Z_PARAM_RANGE
)

fig.update_layout(
    autosize=True,
    width=None,
    height=None
)
fig.write_html("htmls/mean2d_brightness_temperature.html")

In [ ]:
!ls htmls

In [ ]:
len(list(range(50))[::2])

In [ ]:
#@title Volumetric Heatmap of Brightness Temperature, Redshift Slider


X, Y, Z = np.mgrid[0:BOX_PARAM//2, 0:BOX_PARAM//2, 0:BOX_PARAM//2]
X, Y, Z = X.flatten(), Y.flatten(), Z.flatten()

fig = go.Figure()

cutoffmin = dTb_min

reduced_Z_PARAM_RANGE = Z_PARAM_RANGE[::5]

for z in reduced_Z_PARAM_RANGE:
  values = dTb_array[z]
  downsampled_values = values[::2,::2,::2]
  fig.add_trace(go.Volume(
      x=X, y=Y, z=Z,
      value=downsampled_values.flatten(), # np.where(values < 10, np.nan, values).flatten(),
      isomin=cutoffmin,
      isomax=dTb_max,
      opacity=0.2, # needs to be small to see through all surfaces
      surface_count=5, # needs to be a large number for good volume rendering
      colorscale=mincut_colorscale("Rainbow", dTb_min, dTb_max, cutoffmin), #Rainbow #"Turbo"  #"Jet" #"Agsunset"
      colorbar=dict(
          title=dict(
              text="Brightness Temperature δT_b [mK]",
              side="right",
          ),
          x = 0.95,
      ),
      # showscale=False
      ))

fig.data[0].visible = True

sliders = [dict(
active=len(reduced_Z_PARAM_RANGE)-1,
pad={"t": 50},
steps=[dict(
    method="update",
    args=[
        {"visible": [_z == z for _z in reduced_Z_PARAM_RANGE]},
        {"title": dict(
                        text=f"Redshift z = {z}",
                        y=0.96,
                        x=0.5,
                        xanchor="center",
                        yanchor="middle"
                    )}
    ],
    label=f"z={z}",
) for z in reduced_Z_PARAM_RANGE]
)]

vals = Z[::5]
texts = vals*2
fig.update_layout(
    scene=dict(
        zaxis=dict(
            ticktext= texts,
            tickvals= vals,
            title = dict(text="z [Mpc]")
        ),
        xaxis=dict(
            ticktext= texts,
            tickvals= vals,
            title = dict(text="x [Mpc]")
        ),
        yaxis=dict(
            ticktext= texts,
            tickvals= vals,
            title = dict(text="y [Mpc]")
        )
    ),
    width=700,height=700,
    sliders=sliders
    )

# fig.add_trace(go.Volume(
#     x=X.flatten(),
#     y=Y.flatten(),
#     z=Z.flatten(),
#     value=values.flatten(), # np.where(values < 10, np.nan, values).flatten(),
#     isomin=dTb_min,
#     isomax=cutoffmin,
#     opacity=0.1, # needs to be small to see through all surfaces
#     surface_count=5, # needs to be a large number for good volume rendering
#     colorscale=[[0.0,'rgb(255,0,255)'],[1.0,'rgb(255,0,255)']], #Rainbow #"Turbo"  #"Jet" #"Agsunset"
#     showscale=False,
#     ))

# fig.update_layout(template='plotly_dark')

fig.show()
fig.update_layout(
    autosize=True,
    width=None,
    height=None
)
fig.write_html("htmls/volume_brightness_temperature_redshift_slider.html")

In [ ]:
#@title Time Evolution of Brightness Temperature

X, Y, Z = np.mgrid[0:BOX_PARAM, 0:BOX_PARAM, 0:BOX_PARAM] # Each is a BOX_PARAM^3
U, V = np.mgrid[0:BOX_PARAM, 0:BOX_PARAM]


fig = go.Figure()
cutoffmin = dTb_min

assert len(Z_PARAM_RANGE) <= BOX_PARAM, "len(Z_PARAM_RANGE) must be less than or equal to BOX_PARAM for this code to work"

values = np.zeros((BOX_PARAM,BOX_PARAM,BOX_PARAM))
starts = [0]
for i, redshift in enumerate(reversed(Z_PARAM_RANGE)):
  start=starts[-1]
  end = int(BOX_PARAM*(i+1)/len(Z_PARAM_RANGE))
  starts.append(end) # this end is next start
  index_correction = 1 if (start-end)*len(Z_PARAM_RANGE) < BOX_PARAM else 0
  end += index_correction
  if end > BOX_PARAM: end = BOX_PARAM
  # print(f"start={start}, end={end}, index_correction={index_correction}, end-start={end-start}")
  values[:,:,start:end] = dTb_array[redshift][:,:,0:end-start]
starts.pop()

fig.add_trace(go.Volume(
    x=X.flatten(), y=Y.flatten(), z=Z.flatten(),
    value=values.flatten(), # np.where(values < 10, np.nan, values).flatten(),
    isomin=cutoffmin,
    isomax=dTb_max,
    opacity=0.2, # needs to be small to see through all surfaces
    surface_count=5, # needs to be a large number for good volume rendering
    colorscale=mincut_colorscale("Rainbow", dTb_min, dTb_max, cutoffmin), #Rainbow #"Turbo"  #"Jet" #"Agsunset"
    colorbar=dict(
        title=dict(
            text="Brightness Temperature δT_b [mK]",
            side="right",
        ),
        x = 0.95,
    ),

      # showscale=False
      ))



# fig.add_trace(go.Scatter3d(
#     x=np.zeros(len(Z_PARAM_RANGE)+1),
#     y=np.zeros(len(Z_PARAM_RANGE)+1),
#     z=np.linspace(0,BOX_PARAM,len(Z_PARAM_RANGE)+1),
#     # marker=dict(
#     #     size=4,
#     #     color=z,
#     #     colorscale='Viridis',
#     # ),
#     line=dict(
#         color='darkblue',
#         width=2
#     )
# ))
fig.update_layout(
    title=dict(
        text=f"Time Evolution of Brightness Temperature",
        y=0.91,
        x=0.5,
        xanchor="center",
        yanchor="middle"
    ),
    width=700,height=700,
    scene=dict(
        zaxis=dict(
            ticktext= list(reversed(Z_PARAM_RANGE)),
            tickvals= starts,
            title = dict(text="Redshift z")
        )
    )
    )
fig.show()

fig.update_layout(
    autosize=True,
    width=None,
    height=None
)
fig.write_html("htmls/Evolution_of_Brightness_Temperature_Slice.html")

# fig.update_layout(template='plotly_dark')

## Lightcone

In [ ]:
#@title Simulation for Light Cone
BOX_PARAM, BOX_LEN = 50, 50
z_cone_full_range = np.round(t2c.lightcone.redshifts_at_equal_comoving_distance(5,35,BOX_PARAM,BOX_LEN),4)

z_min, z_max = 7, 10
z_cone_range = np.array([z for z in z_cone_full_range if z_min <= z <= z_max])
print(len(z_cone_range), z_cone_range)

if input("Continue? [y]: ").lower().strip() == 'y':
    dTb_cone = np.empty((len(z_cone_range),BOX_PARAM,BOX_PARAM))
    for i, Z_PARAM in enumerate(z_cone_range):
        filename = f"dTb_BOX_PARAM_{BOX_PARAM}_Z_PARAM_{Z_PARAM}.npy"
        # print(f"before : os.path.exists({filename}) {os.path.exists(filename)}")
        if not os.path.exists(filename):
            print(f"Simulating [{i+1}/{len(z_cone_range)}] z={z_cone_range[i]:.4f} ...",end="")
            box = get_brightness_temp_box(
                          box=BOX_PARAM,
                          box_len=BOX_LEN,
                          redshift=Z_PARAM,
                          hubble=0.69,
                          matter=0.31,
                          baryon=0.02)
            np.save(filename, box)
            print("\b\b\b\b. Completed and saved.")
            if i==0: dTb_max, dTb_min = np.nanmax(box), np.nanmin(box)
            if i>0: dTb_max, dTb_min = max(np.nanmax(box), dTb_max), min(np.nanmin(box), dTb_min)
        else:
            box = np.load(filename)
            print(f"[{i+1}/{len(z_cone_range)}] z={z_cone_range[i]:.4f} loaded from file.")
        # print(f"after : os.path.exists({filename}) {os.path.exists(filename)}")
        dTb_cone[i,:,:]=box[i%BOX_PARAM]

cone_filename = f"dTb_cone_BOX_PARAM_{BOX_PARAM}_z_min_{z_min}_z_max_{z_max}.npy"

In [ ]:
#@title Light Cone Brightness Temperature Histogram

dTb_max, dTb_min = np.nanmax(dTb_cone), np.nanmin(dTb_cone)
fig = go.Figure()
cutoffmin = dTb_min

fig.add_trace(go.Heatmap(
        z=dTb_cone[:,:,0].T,
        hoverinfo = "z",
        zmax = dTb_max,
        zmin = dTb_min,
        colorscale = "Turbo"))

tickvals = list(range(0,len(z_cone_range),len(z_cone_range)//10))
ticktext = [f"{z:.4f}" for z in z_cone_range[tickvals]]

fig.update_layout(
    title=dict(
        text=f"Light Cone",
        y=0.91,
        x=0.5,
        xanchor="center",
        yanchor="middle"
    ),
    width=950,height=450,

    xaxis=dict(
        title=dict(text=r"Redshift z"),
        tickvals = tickvals,
        ticktext= ticktext,
        ),
    yaxis=dict(
        title=dict(text=r"y [Mpc]"),
        ),
    )
fig.show()
fig.update_layout(
    autosize=True,
    width=None,
    height=None
)
fig.write_html("htmls/heatmap_light_cone.html")

# fig.update_layout(template='plotly_dark')

In [ ]:
#@title Average Volumetric Light Cone Brightness Temperature

dTb_max, dTb_min = np.nanmax(dTb_cone), np.nanmin(dTb_cone)
# How many slices to average?
avgrange = 10
coneoutput_len = len(dTb_cone)//10
dTb_cone = np.array(dTb_cone)
downsampled_dTbcone = np.zeros((coneoutput_len,BOX_PARAM,BOX_PARAM))
for i in range(coneoutput_len):
  start = i*avgrange
  end = start+avgrange if start+avgrange < len(dTb_cone) else len(dTb_cone)
  downsampled_dTbcone[i,:,:] = np.mean(dTb_cone[start:end,:,:],axis=0)

X, Y, Z = np.mgrid[0:coneoutput_len, 0:BOX_PARAM, 0:BOX_PARAM] # Each is a BOX_PARAM^3
# U, V = np.mgrid[0:BOX_PARAM, 0:BOX_PARAM]

fig = go.Figure()
cutoffmin = dTb_min

fig.add_trace(go.Volume(
    x=X.flatten(), y=Y.flatten(), z=Z.flatten(),
    value=downsampled_dTbcone.flatten(), # np.where(values < 10, np.nan, values).flatten(),
    isomin=cutoffmin,
    isomax=dTb_max,
    opacity=0.2, # needs to be small to see through all surfaces
    surface_count=5, # needs to be a large number for good volume rendering
    colorscale="Rainbow", #mincut_colorscale("Rainbow", dTb_min, dTb_max, cutoffmin), #Rainbow #"Turbo"  #"Jet" #"Agsunset"
    colorbar=dict(
        title=dict(
            text="Brightness Temperature δT_b [mK]",
            side="right",
        ),
        x = 0.95,
    ),
    # showscale=False
))

tickvals = list(range(0,len(z_cone_range),len(z_cone_range)//10))
ticktext = [f"{z:.4f}" for z in z_cone_range[tickvals]]
tickvals = [t//10 for t in tickvals]

fig.update_layout(
    title=dict(
        text=f"Light Cone",
        y=0.91,
        x=0.5,
        xanchor="center",
        yanchor="middle"
    ),
    width=600,height=600,

    scene=dict(
        xaxis=dict(
            ticktext= ticktext,
            tickvals= tickvals,
            title = dict(text="Redshift z")
        ),
        yaxis=dict(
            # ticktext= list(Z_PARAM_RANGE),
            # tickvals= starts,
            title = dict(text="x")
        ),
        zaxis=dict(
            # ticktext= ticktext,
            # tickvals= tickvals,
            title = dict(text="y")
        )
    )
    )


# fig.update_layout(scene_aspectmode='cube')
fig.show()
fig.update_layout(
    autosize=True,
    width=None,
    height=None
)
fig.write_html("htmls/volume_light_cone.html")


# fig.update_layout(template='plotly_dark')

In [13]:
%%script true
#@title Exact Volumetric Light Cone Brightness Temperature

X, Y, Z = np.mgrid[0:len(dTb_cone), 0:BOX_PARAM, 0:BOX_PARAM] # Each is a BOX_PARAM^3
# U, V = np.mgrid[0:BOX_PARAM, 0:BOX_PARAM]

fig = go.Figure()
cutoffmin = dTb_min

fig.add_trace(go.Volume(
    x=X.flatten(), y=Y.flatten(), z=Z.flatten(),
    value=np.array(dTb_cone).flatten(), # np.where(values < 10, np.nan, values).flatten(),
    isomin=cutoffmin,
    isomax=dTb_max,
    opacity=0.2, # needs to be small to see through all surfaces
    surface_count=5, # needs to be a large number for good volume rendering
    colorscale="Rainbow", #mincut_colorscale("Rainbow", dTb_min, dTb_max, cutoffmin), #Rainbow #"Turbo"  #"Jet" #"Agsunset"
    colorbar=dict(
        title=dict(
            text="Brightness Temperature δT_b [mK]",
            side="right",
        ),
        x = 0.95,
    ),
    # showscale=False
))

tickvals = list(range(0,len(z_cone_range),len(z_cone_range)//10))
ticktext = [f"{z:.4f}" for z in z_cone_range[tickvals]]

fig.update_layout(
    title=dict(
        text=f"Light Cone",
        y=0.91,
        x=0.5,
        xanchor="center",
        yanchor="middle"
    ),
    width=600,height=600,

    scene=dict(
        xaxis=dict(
            # ticktext= list(Z_PARAM_RANGE),
            # tickvals= starts,
            title = dict(text="Redshift z")
        ),
        yaxis=dict(
            # ticktext= list(Z_PARAM_RANGE),
            # tickvals= starts,
            title = dict(text="x")
        ),
        zaxis=dict(
            # ticktext= list(Z_PARAM_RANGE),
            # tickvals= starts,
            title = dict(text="y")
        )
    )
    )


fig.update_layout(scene_aspectmode='cube')
fig.show()
fig.update_layout(
    autosize=True,
    width=None,
    height=None
)
fig.write_html("htmls/exact_volume_light_cone.html")


# fig.update_layout(template='plotly_dark')

## Frequency and Power

In [ ]:
#@title Global Frequency Spectrum (Global Signal)
def frequency(z): return 1420.40575 / (1 + z)

frequencies = [frequency(z) for z in Z_PARAM_RANGE]
global_means = [np.mean(dTb_array[z]) for z in Z_PARAM_RANGE]

fig = go.Figure(data=go.Scatter(x=frequencies, y=global_means, mode='lines+markers'))

fig.update_layout(
    title="Global Frequency Spectrum",
    xaxis_title="Frequency [MHz]",
    yaxis_title=r"Global Brightness Temperature δT_b [mK] [mK]",
    width=700,
    height=500
)


fig.show()
fig.update_layout(
    autosize=True,
    width=None,
    height=None
)
fig.write_html("htmls/global_frequency_spectrum.html")

In [ ]:
#@title Spherically averaged 1D power spectrum (Raw vs. Mean Subtracted)
fig = go.Figure()

num_z = len(Z_PARAM_RANGE)
cscale_name = "Turbo"
colors = px.colors.sample_colorscale(cscale_name, [i/(num_z - 1) for i in range(num_z)])

# --- 1. Add Raw Data Traces ---
for i, Z_PARAM in enumerate(Z_PARAM_RANGE):
    ps_raw, ks_raw = t2c.power_spectrum_1d(dTb_array[Z_PARAM], kbins=15, box_dims=BOX_PARAM)
    fig.add_trace(go.Scatter(
        x=ks_raw,
        y=ps_raw * ks_raw**3 / 2 / np.pi**2,
        mode='lines',
        line=dict(color=colors[i]),
        name=f'z={Z_PARAM} (Raw)',
        visible=True
    ))

# --- 2. Add Mean Subtracted Traces ---
for i, Z_PARAM in enumerate(Z_PARAM_RANGE):
    ps_sub, ks_sub = t2c.power_spectrum_1d(t2c.subtract_mean_signal(dTb_array[Z_PARAM]), kbins=15, box_dims=BOX_PARAM)
    fig.add_trace(go.Scatter(
        x=ks_sub,
        y=ps_sub * ks_sub**3 / 2 / np.pi**2,
        mode='lines',
        line=dict(color=colors[i]),
        name=f'z={Z_PARAM} (Subtracted)',
        visible=False
    ))

# --- 3. Add Dummy Trace for Redshift Colorbar ---
# We use +1 to the visibility list later to keep this colorbar always visible
fig.add_trace(go.Scatter(
    x=[None], y=[None],
    mode='markers',
    marker=dict(
        colorscale=cscale_name,
        cmin=min(Z_PARAM_RANGE),
        cmax=max(Z_PARAM_RANGE),
        showscale=True,
        colorbar=dict(title="Redshift (z)", thickness=20, x=1.05)
    ),
    showlegend=False
))

# --- 4. Create Dropdown Menu ---
# Note: + [True] at the end of visibility arrays ensures the colorbar trace stays active
show_raw = [True] * num_z + [False] * num_z + [True]
show_subtracted = [False] * num_z + [True] * num_z + [True]

fig.update_layout(
    updatemenus=[
        dict(
            active=0,
            buttons=list([
                dict(label="Raw Data",
                     method="update",
                     args=[{"visible": show_raw},
                           {"title": "Spherically averaged power spectrum (Raw)"}]),
                dict(label="Mean Subtracted",
                     method="update",
                     args=[{"visible": show_subtracted},
                           {"title": "Spherically averaged power spectrum (Mean Subtracted)"}]),
            ]),
            direction="down",
            pad={"r": 10, "t": 10},
            showactive=True,
            x=0.025,
            xanchor="left",
            y=1.15,
            yanchor="top"
        )
    ],
    title='Spherically averaged power spectrum (Raw)',
    xaxis_title='k (Mpc<sup>-1</sup>)',
    yaxis_title='P(k) k<sup>3</sup>/(2π<sup>2</sup>)',
    xaxis_type='log',
    yaxis_type='log',
    width=700,
    height=500,
    showlegend=False # Colorbar provides the mapping, so legend is redundant
)

fig.show()
fig.update_layout(
    autosize=True,
    width=None,
    height=None
)
fig.write_html("htmls/spherically_averaged_power_spectrum.html")

In [ ]:
#@title Cylindrically Averaged 2D Power Spectrum Slider (Global Color Scaling)

# --- 1. PRE-CALCULATE DATA & FIND GLOBAL LIMITS ---
print("Pre-calculating power spectra and finding global min/max. Please wait...")

N = dTb_array[Z_PARAM_RANGE[0]].shape[0]
L = BOX_PARAM
k_fund = 2 * np.pi / L
k_nyq = np.pi * N / L
custom_kbins = np.linspace(k_fund, k_nyq, N // 2)

ps_results = []
all_values = []

for z in Z_PARAM_RANGE:
    ps, ksperp, kspar = t2c.power_spectrum_2d(
        dTb_array[z],
        kbins=[custom_kbins, custom_kbins],
        box_dims=BOX_PARAM
    )
    ps_results.append((ps, ksperp, kspar, z))
    # Collect all positive values to find global min/max for the color scale
    all_values.extend(ps[ps > 0].flatten())

global_vmin = np.nanmin(all_values)
global_vmax = np.nanmax(all_values)

# --- 2. SETUP PLOT ---
fig = go.Figure()

for i, (ps, ksperp, kspar, z) in enumerate(ps_results):
    ps_safe = np.copy(ps.T)
    ps_safe[ps_safe <= 0] = np.nan
    log_ps = np.log10(ps_safe)

    fig.add_trace(go.Heatmap(
        x=ksperp,
        y=kspar,
        z=log_ps,
        colorscale='Rainbow',
        zmin=np.log10(global_vmin),
        zmax=np.log10(global_vmax),
        # Hide individual colorbars to use one global one
        showscale=False,
        visible=(i == 0)
    ))

# Add a dummy trace to host the global colorbar with log-labeled ticks
fig.add_trace(go.Scatter(
    x=[None], y=[None],
    mode='markers',
    marker=dict(
        colorscale='Rainbow',
        cmin=np.log10(global_vmin),
        cmax=np.log10(global_vmax),
        showscale=True,
        colorbar=dict(
            title='P(k) [mK<sup>2</sup> Mpc<sup>3</sup>]',
            tickvals=np.arange(int(np.floor(np.log10(global_vmin))), int(np.ceil(np.log10(global_vmax))) + 1),
            ticktext=[f'10<sup>{int(v)}</sup>' for v in np.arange(int(np.floor(np.log10(global_vmin))), int(np.ceil(np.log10(global_vmax))) + 1)]
        )
    ),
    hoverinfo='none',
    showlegend=False
))

# --- 3. CREATE SLIDER ---
steps = []
num_z = len(Z_PARAM_RANGE)
for i, z in enumerate(Z_PARAM_RANGE):
    # Visibility: Current heatmap (True), all other heatmaps (False), Colorbar trace (True)
    visibility = [False] * num_z + [True]
    visibility[i] = True

    step = dict(
        method="update",
        args=[
            {"visible": visibility},
            {"title": f'Cylindrically Averaged 2D Power Spectrum (z={z})'}
        ],
        label=f"z={z}"
    )
    steps.append(step)

sliders = [dict(active=0, pad={"t": 50}, steps=steps)]

fig.update_layout(
    title=dict(text=f'Cylindrically Averaged 2D Power Spectrum (z={Z_PARAM_RANGE[0]})', x=0.5, xanchor="center"),
    xaxis=dict(title='k_perp [Mpc<sup>-1</sup>]', type='log'),
    yaxis=dict(title='k_parallel [Mpc<sup>-1</sup>]', type='log'),
    width=750,
    height=650,
    sliders=sliders
)

fig.show()
fig.update_layout(
    autosize=True,
    width=None,
    height=None
)
fig.write_html("htmls/cylindrically_averaged_power_spectrum_redshift_slider.html")

In [ ]:
#@title 3D Stacked Cylindrical Power Spectra (Focus Slider)

# --- 1. PRE-CALCULATE DATA ---
print("Pre-calculating power spectra for all redshifts. Please wait...")

N = dTb_array[Z_PARAM_RANGE[0]].shape[0]
L = BOX_PARAM
k_fund = 2 * np.pi / L
k_nyq = np.pi * N / L
custom_kbins = np.linspace(k_fund, k_nyq, N // 2)

fig = go.Figure()

num_z = len(Z_PARAM_RANGE)
cscale_name = "Turbo"
colors = px.colors.sample_colorscale(cscale_name, [i/(num_z - 1) for i in range(num_z)])

# Store surfaces
for i, z in enumerate(Z_PARAM_RANGE):
    ps, ksperp, kspar = t2c.power_spectrum_2d(
        dTb_array[z],
        kbins=[custom_kbins, custom_kbins],
        box_dims=BOX_PARAM
    )

    ps_safe = np.copy(ps.T)
    ps_safe[ps_safe <= 0] = np.nan
    log_ps = np.log10(ps_safe)

    current_color = colors[i]
    solid_colorscale = [[0, current_color], [1, current_color]]

    fig.add_trace(go.Surface(
        x=ksperp,
        y=kspar,
        z=log_ps,
        surfacecolor=np.zeros(log_ps.shape),
        colorscale=solid_colorscale,
        name=f'z={z}',
        showscale=False,
        opacity=1.0, # Start with 'All' view
        lighting=dict(ambient=0.6, diffuse=0.9, roughness=0.5, specular=0.2)
    ))

# --- 2. ADD REDSHIFT COLORBAR ---
fig.add_trace(go.Scatter3d(
    x=[None], y=[None], z=[None],
    mode='markers',
    marker=dict(
        colorscale=cscale_name,
        cmin=min(Z_PARAM_RANGE),
        cmax=max(Z_PARAM_RANGE),
        showscale=True,
        colorbar=dict(title="Redshift (z)", thickness=20, x=1.1)
    ),
    showlegend=False
))

# --- 3. CREATE SLIDER STEPS ---
steps = []

# "All" Step (index 0)
steps.append(dict(
    method="update",
    args=[{"opacity": [1.0] * num_z + [1.0]}], # Keep colorbar trace visible too
    label="All"
))

# Individual "z" Steps
for i, z in enumerate(Z_PARAM_RANGE):
    # Set target surface to 1.0, others to 0.25, and colorbar trace to 1.0
    opacities = [0.025] * num_z + [1.0]
    opacities[i] = 1.0

    steps.append(dict(
        method="update",
        args=[{"opacity": opacities}],
        label=f"z={z}"
    ))

sliders = [dict(
    active=0,
    currentvalue={"prefix": "Focus: "},
    pad={"t": 50},
    steps=steps
)]

fig.update_layout(
    title=dict(text='3D Cylindrical Power Spectra: Redshift Evolution', x=0.5),
    scene=dict(
        xaxis_title='k_perp',
        yaxis_title='k_parallel',
        zaxis_title='log10(P(k))',
        xaxis_type='log',
        yaxis_type='log'
    ),
    width=1000,
    height=800,
    margin=dict(l=0, r=50, b=0, t=50),
    sliders=sliders
)

fig.show()
fig.update_layout(
    autosize=True,
    width=None,
    height=None
)
fig.write_html("htmls/3d_cylindrical_power_spectra.html")

In [ ]:
!ls htmls